In [ ]:
import os, glob, io, numpy as np
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm
import ipywidgets as widgets
from IPython.display import display

# --- checkpoint (sama dengan cek_model.ipynb) ---
CANDS = ["EfficientNet-B0-head-ft.pth", os.path.join("..","EfficientNet-B0-head-ft.pth"),
         "EfficientNet-B0-triplet-v4.pth", os.path.join("..","EfficientNet-B0-triplet-v4.pth")]
CKPT = next((p for p in CANDS if os.path.exists(p)), None) or (glob.glob("*.pth")+glob.glob(os.path.join("..","*.pth")))[0]
DEVICE = torch.device("cpu"); IMG_SIZE = 224

def _load_white_bg(path):
    img = Image.open(path)
    if img.mode in ("RGBA","LA") or (img.mode=="P" and "transparency" in img.info):
        img = img.convert("RGBA"); alpha = img.getchannel("A")
        if alpha.getextrema()[0] < 255:
            black=Image.new("RGBA",img.size,(0,0,0,255)); tr=Image.new("RGBA",img.size,(255,255,255,0))
            img=Image.composite(black,tr,alpha); bg=Image.new("RGBA",img.size,(255,255,255,255)); img=Image.alpha_composite(bg,img)
    return img.convert("RGB")

def autocrop_and_pad(img, pad_frac=0.08, ink_thresh=245, size=IMG_SIZE):
    gray=np.array(img.convert("L")); mask=gray<ink_thresh
    if mask.sum()==0: cropped=img.convert("L")
    else:
        ys,xs=np.where(mask); y0,y1=int(ys.min()),int(ys.max()); x0,x1=int(xs.min()),int(xs.max()); h,w=gray.shape
        py=max(2,int((y1-y0+1)*pad_frac)); px=max(2,int((x1-x0+1)*pad_frac))
        y0,y1=max(0,y0-py),min(h-1,y1+py); x0,x1=max(0,x0-px),min(w-1,x1+px)
        cropped=img.convert("L").crop((x0,y0,x1+1,y1+1))
    cw,ch=cropped.size; side=max(cw,ch); canv=Image.new("L",(side,side),255); canv.paste(cropped,((side-cw)//2,(side-ch)//2))
    return canv.resize((size,size),Image.BILINEAR)

_MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1); _STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
def preprocess(path):
    img=autocrop_and_pad(_load_white_bg(path)).convert("RGB")
    arr=np.array(img,dtype=np.float32)/255.0; t=torch.from_numpy(arr).permute(2,0,1); return (t-_MEAN)/_STD

class EfficientNetEmbedder(nn.Module):
    def __init__(self, embedding_dim=512, dropout=0.3):
        super().__init__()
        self.backbone=nn.Sequential(tvm.efficientnet_b0(weights=None).features)
        self.head=nn.Sequential(nn.Flatten(), nn.Linear(1280,512), nn.BatchNorm1d(512),
                                nn.ReLU(inplace=True), nn.Dropout(dropout),
                                nn.Linear(512,embedding_dim), nn.BatchNorm1d(embedding_dim))
    def forward(self,x):
        x=self.backbone(x); x=F.adaptive_avg_pool2d(x,1); return F.normalize(self.head(x),p=2,dim=1)

ckpt=torch.load(CKPT, map_location=DEVICE, weights_only=False)
state=ckpt.get("model_state_dict",ckpt) if isinstance(ckpt,dict) else ckpt
EMB=int(ckpt.get("embedding_dim",512)) if isinstance(ckpt,dict) else 512
model=EfficientNetEmbedder(EMB).to(DEVICE); model.load_state_dict(state,strict=True); model.eval()

@torch.no_grad()
def embed(path): return model(preprocess(path).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
def compare(p1,p2,thr):
    s=float(np.dot(embed(p1),embed(p2)))
    return s, ("COCOK — tanda tangan SAMA" if s>=thr else "TIDAK COCOK — tanda tangan BEDA")

def seen_png(path, out_path):
    """Simpan PNG dari citra 224x224 PERSIS yang masuk ke model (setelah autocrop +
    normalisasi dibalik) -- dipakai utk debug: pastikan capture kanvas tidak rusak/kosong
    sebelum menyalahkan model."""
    t = preprocess(path)  # tensor ternormalisasi ImageNet, [3,224,224]
    img = (t*_STD + _MEAN).clamp(0,1).permute(1,2,0).numpy()
    Image.fromarray((img*255).astype("uint8")).save(out_path)
print("Model siap:", os.path.basename(CKPT), "| emb", EMB)

In [ ]:
from ipycanvas import Canvas

CW, CH = 320, 170

CSS = """
.tt-wrap { max-width: 780px; margin: 10px auto 40px; font-family: 'Segoe UI', Roboto, Arial, sans-serif; }
.tt-title { font-size: 22px; font-weight: 700; color: #1a1d21; margin: 0 0 4px; }
.tt-sub { color: #6b7280; font-size: 13px; line-height: 1.5; margin-bottom: 18px; }
.tt-sub code { background:#f1f3f5; padding:1px 6px; border-radius:5px; font-size:12px; }

.tt-cards { align-items: flex-start !important; gap: 16px; }
.tt-card { background:#fff; border:1px solid #e5e7eb; border-radius:14px;
           padding:14px 16px 16px; box-shadow:0 1px 3px rgba(0,0,0,.06); }
.tt-card-title { font-weight:600; font-size:13px; color:#374151; margin-bottom:8px; }
.tt-canvas, .tt-canvas canvas { border-radius:10px !important; border:1.5px solid #d1d5db !important; }
.tt-controls { margin-top:10px !important; gap:8px; }
.tt-controls .jupyter-button { border-radius:8px !important; }

.tt-divider { display:flex; align-items:center; justify-content:center; padding: 0 2px; height: 100%; }
.tt-divider span { background:#eef2ff; color:#4f46e5; font-weight:700; font-size:11px;
                   border-radius:999px; padding:7px 9px; border:1px solid #e0e7ff; white-space:nowrap; }

.tt-actions { margin-top:20px !important; gap:18px; align-items:center !important;
              background:#f9fafb; border:1px solid #eef0f2; border-radius:12px; padding:12px 16px; }
.tt-actions .widget-label { font-size:13px !important; color:#374151 !important; }
.tt-actions .jupyter-button { border-radius:10px !important; font-weight:600 !important; padding:6px 22px !important; }

.tt-result { border-radius:12px; padding:16px 18px; border-left:5px solid; margin-bottom:4px; }
.tt-result.ok { background:#f0fdf4; border-color:#16a34a; }
.tt-result.no { background:#fef2f2; border-color:#dc2626; }
.tt-result-verdict { font-size:18px; font-weight:700; }
.tt-result-meta { color:#6b7280; font-size:12.5px; margin-top:4px; }
.tt-bar-track { margin-top:10px; height:10px; border-radius:20px; background:#e5e7eb; overflow:hidden; }
.tt-bar-fill { height:100%; border-radius:20px; }

.tt-debug { margin-top:14px !important; border:1.5px dashed #d1d5db !important; border-radius:12px !important;
            padding:12px 14px !important; background:#fafafa !important; }
.tt-debug-title { font-size:11px; color:#9ca3af; font-weight:700; text-transform:uppercase;
                  letter-spacing:.03em; margin-bottom:8px; }
.tt-debug img { border-radius:8px; border:1px solid #e5e7eb; }
.tt-debug-meta { color:#9ca3af; font-size:11px; margin-top:6px; font-family: ui-monospace, monospace; }
"""

class SigInput:
    """Satu slot tanda tangan: bisa DIGAMBAR di kanvas atau di-UPLOAD."""
    def __init__(self, label):
        self.canvas = Canvas(width=CW, height=CH, sync_image_data=True)
        self.canvas.add_class("tt-canvas")
        self.drawing = False; self.points = []; self.has_ink = False
        self._clear()
        self.canvas.on_mouse_down(self._down)
        self.canvas.on_mouse_move(self._move)
        self.canvas.on_mouse_up(self._up)
        self.canvas.on_mouse_out(self._up)
        self.upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload")
        self.clear_btn = widgets.Button(description="Hapus", icon="trash", layout=widgets.Layout(width="90px"))
        self.clear_btn.on_click(lambda _: self._clear())
        controls = widgets.HBox([self.clear_btn, self.upload])
        controls.add_class("tt-controls")
        self.box = widgets.VBox([widgets.HTML(f"<div class='tt-card-title'>{label}</div>"),
                                 self.canvas, controls])
        self.box.add_class("tt-card")
    def _clear(self):
        self.canvas.fill_style = "white"; self.canvas.fill_rect(0, 0, CW, CH)
        self.canvas.stroke_style = "black"; self.canvas.line_width = 3
        self.canvas.line_cap = "round"; self.canvas.line_join = "round"
        self.has_ink = False; self.points = []
    def _down(self, x, y): self.drawing = True; self.points = [(x, y)]
    def _move(self, x, y):
        if self.drawing:
            # Gambar SELURUH goresan-sejauh-ini sbg SATU polyline (bukan potongan
            # kecil per gerakan mouse). Kalau digambar potongan-potongan dgn
            # begin_path()+stroke() terpisah, ujung bulat tiap potongan saling
            # menumpuk di setiap sambungan dan tepi antialiasing-nya menumpuk
            # opacity-nya -> makin banyak titik (makin lama menggambar) makin
            # gelap/tebel kelihatannya. stroke_lines() menghitung ulang seluruh
            # bentuk sekali jalan, jadi tak ada penumpukan.
            self.points.append((x, y))
            if len(self.points) >= 2:
                self.canvas.stroke_lines(self.points)
            self.has_ink = True
    def _up(self, x=None, y=None): self.drawing = False
    def to_png(self, path):
        # utamakan yang DIGAMBAR; kalau kanvas kosong, pakai UPLOAD
        if self.has_ink:
            try:
                arr = self.canvas.get_image_data()
                Image.fromarray(arr.astype("uint8")).convert("RGB").save(path)
                return True
            except Exception:
                pass
        v = self.upload.value
        if v:
            item = v[0] if isinstance(v, (list, tuple)) else list(v.values())[0]
            open(path, "wb").write(bytes(item["content"])); return True
        return False

sig1 = SigInput("Tanda Tangan 1")
sig2 = SigInput("Tanda Tangan 2")

divider = widgets.HTML("<div class='tt-divider'><span>VS</span></div>")
cards = widgets.HBox([sig1.box, divider, sig2.box])
cards.add_class("tt-cards")

thr = widgets.FloatSlider(value=0.50, min=0.20, max=0.90, step=0.01, description="Threshold", readout_format=".2f")
btn = widgets.Button(description="Bandingkan", button_style="primary", icon="check")
actions = widgets.HBox([thr, btn])
actions.add_class("tt-actions")

out = widgets.Output()

def _on_click(_):
    out.clear_output()
    with out:
        ok1 = sig1.to_png("_c1.png"); ok2 = sig2.to_png("_c2.png")
        if not ok1 or not ok2:
            display(widgets.HTML(
                "<div class='tt-result no'>"
                "<div class='tt-result-verdict' style='color:#dc2626'>Lengkapi dulu</div>"
                "<div class='tt-result-meta'>Isi KEDUA tanda tangan (gambar di kanvas atau upload) "
                "sebelum membandingkan.</div></div>"))
            return

        s, verd = compare("_c1.png", "_c2.png", thr.value)
        pct = max(0, min(100, s*100))
        ok = s >= thr.value
        color = "#16a34a" if ok else "#dc2626"
        cls = "ok" if ok else "no"
        display(widgets.HTML(
            f"<div class='tt-result {cls}'>"
            f"<div class='tt-result-verdict' style='color:{color}'>{verd}</div>"
            f"<div class='tt-result-meta'>Kemiripan cosine = <b>{s:.4f}</b> &middot; threshold = {thr.value:.2f}</div>"
            f"<div class='tt-bar-track'><div class='tt-bar-fill' style='width:{pct:.1f}%;background:{color}'></div></div>"
            f"<div class='tt-result-meta' style='margin-top:6px'>{pct:.1f}% kemiripan</div>"
            f"</div>"))

        # --- Debug (sekunder): tampilkan PERSIS gambar 224x224 yang masuk ke model,
        # supaya kelihatan kalau capture kanvas rusak/kosong, sebelum menyalahkan model. ---
        seen_png("_c1.png", "_seen1.png"); seen_png("_c2.png", "_seen2.png")
        raw1 = np.array(Image.open("_c1.png").convert("L"))
        raw2 = np.array(Image.open("_c2.png").convert("L"))
        debug_box = widgets.VBox([
            widgets.HTML("<div class='tt-debug-title'>Debug &middot; yang dilihat model (crop otomatis 224&times;224)</div>"),
            widgets.HBox([
                widgets.Image(value=open("_seen1.png","rb").read(), width=120),
                widgets.Image(value=open("_seen2.png","rb").read(), width=120),
            ]),
            widgets.HTML(f"<div class='tt-debug-meta'>capture 1: {raw1.shape}, ink={int((raw1<200).sum())}px"
                        f" &nbsp;|&nbsp; capture 2: {raw2.shape}, ink={int((raw2<200).sum())}px</div>"),
        ])
        debug_box.add_class("tt-debug")
        display(debug_box)
btn.on_click(_on_click)

header = widgets.HTML(
    "<div class='tt-title'>Bandingkan 2 Tanda Tangan</div>"
    "<div class='tt-sub'>Opsi B &middot; dijalankan via <code>voila compare_app.ipynb</code>. "
    "Gambar di kanvas <b>atau</b> upload, lalu klik <b>Bandingkan</b>. Model &amp; logika identik dengan notebook.</div>")

root = widgets.VBox([header, cards, actions, out])
root.add_class("tt-wrap")

display(widgets.HTML(f"<style>{CSS}</style>"))
display(root)